# Standalone JIT

Compiling a hand-written NumPy function to native Rust, and getting its exact Jacobian with it.

## What the JIT does

FastSim traces a Python function once, records the operations it performs, and lowers them to the scalar SSA form the engine uses internally. From then on the function runs as native Rust — the Python callable is not touched again.

Blocks that take a custom function do this automatically. `fastsim.jit` is the same machinery on its own, for any function $f(x, t)$:

- `jit(f)` — the traced function, callable like the original.
- `jacobian(f)` — the exact $\partial f / \partial x$, differentiated from the recorded operations rather than approximated by finite differences.

Because the traced form is IR, a model built on such a function is not stuck in Python: it lowers to C and exports to an FMU like any native block.

## The Function

The Robertson kinetics problem — three species, rate constants nine orders of magnitude apart:

$$\begin{aligned}
\dot{y}_1 &= -k_1 y_1 + k_2 y_2 y_3\\
\dot{y}_2 &= k_1 y_1 - k_2 y_2 y_3 - k_3 y_2^2\\
\dot{y}_3 &= k_3 y_2^2
\end{aligned}$$

with $k_1 = 0.04$, $k_2 = 10^4$, $k_3 = 3\cdot 10^7$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Apply the FastSim docs matplotlib style
plt.style.use('../fastsim_docs.mplstyle')

from fastsim import Simulation, Connection
from fastsim.blocks import ODE, Scope
from fastsim.solvers import GEAR52A
from fastsim.jit import jit, jacobian

## System Parameters

In [ ]:
k1, k2, k3 = 0.04, 1.0e4, 3.0e7

y0 = np.array([1.0, 0.0, 0.0])   # all of species 1 to start with

## The Right-Hand Side

Written by hand, in the way you would write it anyway.

In [ ]:
def robertson(y, t):
    return np.array([
        -k1 * y[0] + k2 * y[1] * y[2],
         k1 * y[0] - k2 * y[1] * y[2] - k3 * y[1] ** 2,
                                        k3 * y[1] ** 2,
    ])

## Compiling It

`jit` returns a callable with the same signature. The trace happens on the first call, or eagerly if you pass `n_x`.

In [ ]:
f = jit(robertson)

probe = np.array([0.9, 2.2e-5, 0.1])
print("python:", robertson(probe, 0.0))
print("jit   :", f(probe, 0.0))

The tape is specialised to the length it was traced with. A call with a different length re-traces:

In [ ]:
total = jit(lambda x, t: x.sum())

print(total(np.arange(8.0)))
print(total(np.array([1.0, 2.0])))

## Jacobians

`jacobian` differentiates the traced operations, so one pass yields the whole matrix — no step size to pick, and no extra evaluations of $f$.

In [ ]:
J = jacobian(robertson)

np.set_printoptions(precision=4, linewidth=100)
print(J(probe, 0.0))

## Verification

The Robertson Jacobian can be written down by hand:

$$\frac{\partial f}{\partial y} = \begin{pmatrix}
-k_1 & k_2 y_3 & k_2 y_2\\
k_1 & -k_2 y_3 - 2k_3 y_2 & -k_2 y_2\\
0 & 2 k_3 y_2 & 0
\end{pmatrix}$$

In [ ]:
def J_analytic(y):
    return np.array([
        [-k1,  k2 * y[2],                   k2 * y[1]],
        [ k1, -k2 * y[2] - 2 * k3 * y[1],  -k2 * y[1]],
        [0.0,  2 * k3 * y[1],               0.0],
    ])

err = np.max(np.abs(J(probe, 0.0) - J_analytic(probe)))
print(f"worst difference from the analytic Jacobian: {err:.3e}")

## Using It in a Model

The same function as the right-hand side of an `ODE` block. Robertson is stiff, so it wants an implicit solver — `GEAR52A` here.

In [ ]:
kinetics = ODE(robertson, initial_value=y0)
sco = Scope(labels=["y1", "y2", "y3"], sampling_period=1e-2)

sim = Simulation(
    blocks=[kinetics, sco],
    connections=[Connection(kinetics[:3], sco[:3])],
    Solver=GEAR52A,
    dt=1e-4,
    tolerance_lte_abs=1e-10,
    tolerance_lte_rel=1e-8,
)

sim.run(40.0)

## Results

Species 2 stays six orders of magnitude below the others, so it is scaled up to be visible at all — the usual way this solution is plotted.

In [ ]:
t, [y1, y2, y3] = sco.read()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t, y1, label="y₁")
ax.plot(t, y2 * 1e4, label="y₂ × 10⁴")
ax.plot(t, y3, label="y₃")
ax.set_xlabel("time [s]")
ax.set_ylabel("concentration")
ax.legend()
plt.show()

Mass is conserved, which is the easiest thing to check on this system: $y_1 + y_2 + y_3 = 1$ for all time.

In [ ]:
total_mass = y1 + y2 + y3
print(f"worst deviation from unit mass: {np.max(np.abs(total_mass - 1.0)):.3e}")

## What Traces

Everything the tracer sees has to be an operation on the values, not a decision about them. When it cannot trace something it says so, and says what to write instead.

In [ ]:
probes = {
    "arithmetic":      lambda x, t: np.array([x[1], -x[0]]),
    "np.where":        lambda x, t: np.array([x[1], np.where(x[0] > 0, -x[0], 0.0)]),
    "np.sin / np.exp": lambda x, t: np.array([np.sin(x[0]), np.exp(-x[1])]),
    "np.linalg.norm":  lambda x, t: np.array([np.linalg.norm(x), 0.0]),
    "matrix product":  lambda x, t: np.array([[0.0, 1.0], [-1.0, -0.2]]) @ x,
    "np.clip":         lambda x, t: np.clip(x, -1.0, 1.0),
    "python if":       lambda x, t: np.array([x[1], -x[0] if x[0] > 0 else 0.0]),
    "float() cast":    lambda x, t: np.array([x[1], -float(x[0])]),
}

width = max(len(k) for k in probes)
for name, fn in probes.items():
    try:
        jit(fn)(np.array([1.0, 2.0]), 0.0)
        print(f"  {name:<{width}}  traced")
    except TypeError as e:
        print(f"  {name:<{width}}  {str(e).split('.')[0]}")

A Python `if` on a traced value is the one people hit first. Rewrite it as `np.where` and it traces.